# ResNet34-UNet road extraction on Kaggle — DeepGlobe, 300 epochs, tracked with MLflow

[GohVh/resnet34-unet](https://github.com/GohVh/resnet34-unet): a U-Net decoder on an
ImageNet-pretrained ResNet34 encoder. The network is upstream's `UnetResnet34`, imported
unmodified from a pinned git submodule; this repo adds the DeepGlobe data pipeline, a
training loop sized to get through 300 epochs in a few hours, and MLflow tracking.

**Before you run anything**

1. *Settings → Accelerator* → **GPU T4 x2** — both GPUs train, which roughly halves the
   run — or P100. Switch **Internet on**: the dependency install, the model's submodule
   and its ImageNet weights all download.
2. *Add Input* → the **DeepGlobe Road Extraction Dataset**
   (`balraj98/deepglobe-road-extraction-dataset`).
3. Optional: *Add-ons → Secrets* → `MLFLOW_TRACKING_URI` (plus
   `MLFLOW_TRACKING_USERNAME` / `MLFLOW_TRACKING_PASSWORD`) to log to a remote tracking
   server. With no secret, everything goes to a local file store under
   `/kaggle/working/mlruns`, which is saved as notebook output.

There is no CUDA kernel to build — that was RFM-UNet, which lives on the `rfmunet-kaggle`
branch. Setup takes a few minutes, then step 6 trains; **Save & Run All** runs the whole
notebook unattended.

## 1. What hardware did we get?

In [ ]:
import os, subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print(f"torch          {torch.__version__}")
print(f"cuda runtime   {torch.version.cuda}")
print(f"cpu cores      {os.cpu_count()}")
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    major, minor = torch.cuda.get_device_capability(i)
    props = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {props.name}  compute {major}.{minor}  {props.total_memory / 1024**3:.1f} GB")
if n_gpus == 0:
    raise SystemExit("No GPU. Set Settings -> Accelerator -> GPU, then rerun.")

# Train on every GPU we got: DDP over a T4 x2 roughly halves each epoch. The ranks
# split the CPU cores for their DataLoader workers. Later cells inherit all of these.
os.environ["GPUS"] = str(n_gpus)
os.environ["STRATEGY"] = "ddp" if n_gpus > 1 else "auto"
os.environ["NUM_WORKERS"] = str(max(1, (os.cpu_count() or 2) // n_gpus))
os.environ["PYTHONUNBUFFERED"] = "1"  # stream the training log into the notebook as it happens
print({k: os.environ[k] for k in ("GPUS", "STRATEGY", "NUM_WORKERS")})

## 2. Code and dependencies

The repo can arrive three ways: you are already running inside it, it is attached as a
Kaggle Dataset, or it gets cloned. All three end with `REPO` as the working directory,
and the cell refuses to continue unless the entry points are really there.

The clone is pinned to the **`resnet34unet-kaggle`** branch — `main` still holds the old
U-Net notebook and `rfmunet-kaggle` the RFM-UNet pipeline. Override with the `REPO_BRANCH`
environment variable.

The network's code is GohVh's repository itself, a git submodule at
`third_party/resnet34-unet` pinned to the commit this pipeline was built against. Upstream
ships no licence, so it is fetched from GitHub rather than copied into this repo.

In [ ]:
from pathlib import Path
import shutil, subprocess, os

REPO_URL = "https://github.com/KorewaLidesu/unet-road-segmentation.git"
# The pipeline lives on this branch; main still holds the old U-Net notebook.
REPO_BRANCH = os.environ.get("REPO_BRANCH", "resnet34unet-kaggle")
TARGET = Path("/kaggle/working/ResNet34UNet")
UPSTREAM_URL = "https://github.com/GohVh/resnet34-unet"
UPSTREAM_MODEL = Path("third_party/resnet34-unet/model.py")

# A directory only counts as the repo if the entry points are actually in it.
# Without this check a clone of the wrong branch lands here and the failure
# surfaces two cells later as a missing requirements file.
SENTINELS = ("train_supervision.py", "requirements-kaggle.txt", "tools/prepare_deepglobe.py",
             "config/kaggle/ResNet34UNet.py")


def is_repo(root):
    return all((root / name).exists() for name in SENTINELS)


def missing(root):
    return [name for name in SENTINELS if not (root / name).exists()]


def attached_copy():
    # A repo attached as a Kaggle Dataset, searched one and two levels deep.
    inputs = Path("/kaggle/input")
    if not inputs.exists():
        return None
    for depth in ("*/train_supervision.py", "*/*/train_supervision.py"):
        for hit in sorted(inputs.glob(depth)):
            if is_repo(hit.parent):
                return hit.parent
    return None


def find_repo():
    if is_repo(Path.cwd()):
        return Path.cwd()
    source = attached_copy()
    if source is not None:
        # /kaggle/input is read-only and training writes beside the code.
        if TARGET.exists():
            shutil.rmtree(TARGET)
        shutil.copytree(source, TARGET)
        print(f"copied attached repo from {source}")
        return TARGET
    if TARGET.exists() and not is_repo(TARGET):
        # Left over from an earlier run that cloned the wrong branch.
        print(f"discarding incomplete {TARGET} (missing {missing(TARGET)})")
        shutil.rmtree(TARGET)
    if not TARGET.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                        "--recurse-submodules", "--shallow-submodules",
                        REPO_URL, str(TARGET)], check=True)
    return TARGET


def ensure_upstream(root):
    # Copies made without --recurse-submodules have an empty third_party/resnet34-unet.
    if (root / UPSTREAM_MODEL).exists():
        return
    if (root / ".git").exists():
        subprocess.run(["git", "-C", str(root), "submodule", "update", "--init", "--depth", "1"],
                       check=True)
    else:
        # No git metadata (e.g. an attached Dataset made from a zip), so no pinned
        # commit to check out: take upstream's current main and say so.
        dest = root / UPSTREAM_MODEL.parent
        shutil.rmtree(dest, ignore_errors=True)
        subprocess.run(["git", "clone", "--depth", "1", UPSTREAM_URL, str(dest)], check=True)
        print("note: cloned upstream main, not the pinned submodule commit")


REPO = find_repo()
os.chdir(REPO)
if not is_repo(REPO):
    raise SystemExit(
        f"{REPO} is missing {missing(REPO)}. Set REPO_BRANCH if the pipeline "
        f"has moved off '{REPO_BRANCH}'."
    )
ensure_upstream(REPO)
upstream = subprocess.run(["git", "-C", str(REPO / UPSTREAM_MODEL.parent), "log", "-1", "--format=%h %s"],
                          capture_output=True, text=True).stdout.strip()
print("working in", REPO, f"(branch {REPO_BRANCH})")
print("model code:", UPSTREAM_URL, "@", upstream)
print(sorted(p.name for p in REPO.iterdir() if not p.name.startswith(".")))

In [ ]:
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Build the network once here. That downloads the ImageNet ResNet34 weights into the
# torch cache, so the training ranks find them there instead of each fetching them.
import torch
from geoseg.models.resnet34_unet import ResNet34UNet, upstream_sha

net = ResNet34UNet(num_classes=2).cuda().eval()
print(f"upstream commit  {upstream_sha()}")
print(f"parameters       {sum(p.numel() for p in net.parameters()) / 1e6:.1f} M")
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    out = net(torch.randn(2, 3, 512, 512, device="cuda"))
print(f"512x512 input -> {tuple(out.shape)} logits")
assert out.shape == (2, 2, 512, 512)
del net, out
torch.cuda.empty_cache()

## 3. Prepare the DeepGlobe split

Only DeepGlobe's `train/` folder is labelled — its `valid/` and `test/` folders are the
unlabelled competition holdouts — so the 6226 labelled pairs get split 80/10/10 here. The
split is on source image id *before* tiling, so crops of one scene never straddle splits.
It is the same split, seed and tiling as the RFM-UNet runs, so test numbers compare
directly.

Each 1024×1024 scene becomes four 512×512 tiles (`--mode split`). Validation and testing
score every tile; training cuts one random `CROP_SIZE` window per scene and epoch out of
them. Masks are re-encoded from 0/255 to class indices with **road = 0**, matching
`dpgb.PALETTE` and `tools/metric.py`. Feeding raw 0/255 masks in would make every road
pixel read as `ignore_index=255` and train on background alone — the prep script verifies
this rather than trusting it.

Add `--fraction 0.25` for a fast first pass.

In [ ]:
import os
from pathlib import Path

# Where Kaggle mounts an attached dataset depends on how it was attached, so find
# it instead of hard-coding: DeepGlobe is whatever directory holds a *_sat.jpg with
# a matching *_mask.png. tools/prepare_deepglobe.py rglobs from here, so pointing
# at train/ is both precise and the fastest scan -- valid/ and test/ are the
# unlabelled competition holdouts and have no masks to find anyway.
SEARCH_ROOTS = (Path("/kaggle/input"), Path.home() / ".cache" / "kagglehub")


def find_deepglobe():
    for base in SEARCH_ROOTS:
        if not base.exists():
            continue
        for sat in base.rglob("*_sat.jpg"):
            if sat.with_name(sat.name.replace("_sat.jpg", "_mask.png")).exists():
                return sat.parent
    return None


found = find_deepglobe()
if found is None:
    present = sorted(p.name for p in Path("/kaggle/input").glob("*")) \
        if Path("/kaggle/input").exists() else []
    raise SystemExit(
        "No *_sat.jpg with a matching *_mask.png under " +
        " or ".join(str(r) for r in SEARCH_ROOTS) + ".\n"
        "Attach balraj98/deepglobe-road-extraction-dataset via Add Input -> Datasets, "
        "then rerun this cell.\n"
        f"Currently under /kaggle/input: {present}"
    )

os.environ["RAW_DEEPGLOBE_ROOT"] = str(found)
os.environ["DEEPGLOBE_ROOT"] = "/kaggle/working/data/deepglobe"

print("RAW_DEEPGLOBE_ROOT =", found)
print("labelled pairs here:", len(list(found.glob("*_sat.jpg"))))
print("dataset root holds:", sorted(p.name for p in found.parent.iterdir())[:10])

In [ ]:
!python tools/prepare_deepglobe.py     --src {os.environ["RAW_DEEPGLOBE_ROOT"]}     --dst {os.environ["DEEPGLOBE_ROOT"]}     --mode split --workers 4

In [ ]:
# Eyeball a few prepared pairs and confirm the label encoding.
import cv2, numpy as np, json
import matplotlib.pyplot as plt
from pathlib import Path

root = Path(os.environ["DEEPGLOBE_ROOT"])
print(json.dumps(json.loads((root / ".prepared.json").read_text()), indent=2))

masks = sorted((root / "train_masks").glob("*.png"))[:4]
fig, axes = plt.subplots(2, len(masks), figsize=(4 * len(masks), 8))
for col, mask_path in enumerate(masks):
    img = cv2.cvtColor(cv2.imread(str(root / "train_images" / f"{mask_path.stem}.jpg")), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    axes[0, col].imshow(img); axes[0, col].set_title(mask_path.stem, fontsize=8)
    axes[1, col].imshow(mask == 0, cmap="gray")
    axes[1, col].set_title(f"road px {np.mean(mask == 0):.3f} | values {np.unique(mask).tolist()}", fontsize=8)
    for row in (0, 1):
        axes[row, col].axis("off")
plt.tight_layout(); plt.show()

## 4. MLflow

No configuration needed: without a tracking URI everything lands in
`/kaggle/working/mlruns`, which survives as notebook output. Download it and read it with
`MLFLOW_ALLOW_FILE_STORE=true mlflow ui --backend-store-uri ./mlruns` (current MLflow
refuses a file store without that opt-in; `tools/mlflow_utils.py` sets it here).

If a `MLFLOW_TRACKING_URI` Kaggle Secret exists it is used instead, and credentials come
from the same place. The run id is cached next to the checkpoints, so if the session is
cut short and you rerun, training resumes into the *same* MLflow run rather than starting
a second one.

In [ ]:
from tools import mlflow_utils

found = mlflow_utils.load_kaggle_secrets()
print("secrets picked up:", found or "none (using the local file store)")

os.environ["OUT_ROOT"] = "/kaggle/working"
print("tracking uri:", mlflow_utils.resolve_tracking_uri("/kaggle/working/mlruns"))

## 5. Smoke test

Two training batches and two validation batches, on the same GPUs and strategy as the
real run. This catches config mistakes, shape errors, an OOM at this batch size, a DDP
problem and a broken MLflow target in a minute or two — rather than an hour into the run.

In [ ]:
!rm -rf /kaggle/working/smoke
!OUT_ROOT=/kaggle/working/smoke  MAX_EPOCH=1 LIMIT_TRAIN_BATCHES=2 LIMIT_VAL_BATCHES=2 CHECK_VAL_EVERY_N_EPOCH=1  NUM_SANITY_VAL_STEPS=0  MLFLOW_EXPERIMENT_NAME=ResNet34UNet-smoke MLFLOW_RUN_NAME=smoke  MLFLOW_DEFAULT_DIR=/kaggle/working/smoke/mlruns MLFLOW_SYSTEM_METRICS=0  LOG_PREDICTION_IMAGES=1 PREDICTION_IMAGE_EVERY_N_EPOCHS=1 MAX_TIME=  python train_supervision.py -c config/kaggle/ResNet34UNet.py

## 6. Train — 300 epochs

**What makes 300 epochs fast.** An epoch is one random 256×256 window from each of the
~5,000 training scenes, not a pass over all ~20,000 tiles, so every scene is still seen
every epoch — at a fresh position each time — for a sixteenth of the pixels. On top of
that: both T4s via DDP, 16-bit mixed precision, channels_last, cuDNN benchmark mode,
batch 16 per GPU, and validation every 5 epochs. Upstream's optimiser is kept: AdamW at
1e-3 under a OneCycle schedule stepped every batch.

**How long.** Measured on one RTX 3050 Ti Laptop GPU with four CPU threads for data
loading, then projected to DeepGlobe's ~5,000 training scenes and ~2,500 validation tiles:

| `CROP_SIZE` | per epoch | 300 epochs + validation | T4 x2 (estimate) |
|---|---|---|---|
| **256** (default) | ~42 s | ~4 h | ~2 h |
| 384 | ~94 s | ~8 h | ~4 h |
| 512 | ~166 s | ~14 h | ~7 h |

A T4 is a card of roughly that class, and DDP over two of them roughly halves the time.
The `[speed]` line printed after every epoch, and the `epoch_seconds` metric, give the
real figure from epoch 2 on. Larger windows give the network more road context per
sample, at proportionally more time.

Training runs as a subprocess rather than in-process; that is what makes real DDP work on
a T4 x2, where Lightning's in-notebook strategy is unreliable with dataloader workers.
`MAX_TIME` stops it with a resumable checkpoint before Kaggle's 12-hour limit; rerunning
this cell resumes from `last.ckpt` and continues the same MLflow run. Keep `MAX_EPOCH`
unchanged when resuming — the OneCycle schedule is laid out over the original length.

In [ ]:
!OUT_ROOT=/kaggle/working  DEEPGLOBE_ROOT=/kaggle/working/data/deepglobe  MAX_EPOCH=300 CROP_SIZE=256  TRAIN_BATCH_SIZE=16 VAL_BATCH_SIZE=16 CHECK_VAL_EVERY_N_EPOCH=5  PRECISION=16-mixed MAX_TIME=00:10:30:00  MLFLOW_EXPERIMENT_NAME=ResNet34UNet-DeepGlobe  PREDICTION_IMAGE_EVERY_N_EPOCHS=25  python train_supervision.py -c config/kaggle/ResNet34UNet.py

## 7. Evaluate

Scores the held-out test split with horizontal/vertical flip TTA and writes the predicted
masks. clDice and APLS are computed here — they skeletonise and graph every tile on the
CPU, which is why they are off during training and the slow part of this step
(`--no-topology` skips them) — and all the metrics are logged onto the training run so
they sit beside its curves.

In [ ]:
!OUT_ROOT=/kaggle/working DEEPGLOBE_ROOT=/kaggle/working/data/deepglobe  python road_seg_test.py     -c config/kaggle/ResNet34UNet.py     -o /kaggle/working/results/deepglobe/ResNet34UNet     --rgb -t lr --batch-size 16

## 8. Read the run back

In [ ]:
import mlflow, pandas as pd
import matplotlib.pyplot as plt

mlflow.set_tracking_uri(mlflow_utils.resolve_tracking_uri("/kaggle/working/mlruns"))
runs = mlflow.search_runs(experiment_names=["ResNet34UNet-DeepGlobe"], order_by=["start_time DESC"])
cols = [c for c in ["run_id", "status", "metrics.val_mIoU", "metrics.val_IoU_Road",
                    "metrics.val_F1_Road", "metrics.test_IoU_Road", "metrics.test_clDice",
                    "metrics.test_APLS", "params.crop_size", "params.env/gpu"] if c in runs.columns]
display(runs[cols].head())

In [ ]:
# Training curves for the newest run.
run_id = runs.iloc[0].run_id
client = mlflow.tracking.MlflowClient()

fig, axes = plt.subplots(1, 4, figsize=(21, 4))
panels = [
    ("loss", ["train_loss_epoch", "val_loss"]),
    ("road IoU", ["train_IoU_Road", "val_IoU_Road"]),
    ("mIoU", ["train_mIoU", "val_mIoU"]),
    ("seconds per epoch", ["epoch_seconds"]),
]
for ax, (title, keys) in zip(axes, panels):
    for key in keys:
        history = client.get_metric_history(run_id, key)
        if history:
            ax.plot([h.step for h in history], [h.value for h in history], label=key, marker=".")
    ax.set_title(title); ax.set_xlabel("step"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("prediction previews logged under val_predictions/ in the run's artifacts")

## 9. Keep the outputs

Kaggle keeps `/kaggle/working` (20 GB) as notebook output. Zip the tracking store and keep
the checkpoints (`last.ckpt` and the best one, ~460 MB each with the optimiser state) so the
next session — or your laptop — can pick them up. Save this notebook's output as a Kaggle
Dataset and attach it to continue training later.

In [ ]:
!du -sh /kaggle/working/mlruns /kaggle/working/model_weights 2>/dev/null
!cd /kaggle/working && zip -qr mlruns.zip mlruns && ls -lh mlruns.zip
!ls -lh /kaggle/working/model_weights/deepglobe/ResNet34UNet/
!rm -rf /kaggle/working/data /kaggle/working/smoke   # big and reproducible; drop from the output

### Continuing in a later session

1. Save this session's output as a Kaggle Dataset.
2. In the new notebook, attach it and copy its `model_weights/` and `mlruns/` back to the
   same paths under `/kaggle/working` (attached inputs are read-only, and the cached
   MLflow run id is only reused when the tracking store sits at the same path).
3. Rerun step 6 with the same `MAX_EPOCH`. `last.ckpt` is picked up automatically and the
   metrics continue in the same MLflow run.